In [1]:
# math: Python 标准数学库（本例中保留导入，帮助理解数学公式）
import math

# 导入 PyTorch 主包（张量 Tensor、自动求导、GPU 支持等）
import torch

# 从 torch 导入 nn（neural network）模块，包含 Module、Parameter 等神经网络组件
from torch import nn


In [2]:
# 自定义类 LayerNorm：
# 作用：实现“层归一化（Layer Normalization）”。
# 对每个样本在最后一个维度上做标准化，再进行可学习的缩放（gamma）和偏移（beta）。
class LayerNorm(nn.Module):
    def __init__(self, d_model, eps=1e-10):
        """
        参数说明：
        - d_model: 最后一维特征数（也就是归一化的维度大小）
        - eps: 防止除零的小常数
        """
        # 正确初始化 nn.Module 基类。
        # 旧写法 super(LayerNorm).__init__() 是错误的，会导致初始化异常。
        super().__init__()

        # nn.Parameter: 可训练参数，训练时会被优化器更新。
        # gamma 初始为 1：相当于“先不缩放”。
        self.gamma = nn.Parameter(torch.ones(d_model))

        # beta 初始为 0：相当于“先不平移”。
        self.beta = nn.Parameter(torch.zeros(d_model))

        # 保存 eps，后续归一化时使用
        self.eps = eps

    def forward(self, x):
        """
        前向传播：输入 x -> 标准化 -> 仿射变换。

        输入：
        - x: 形状通常为 [batch_size, seq_len, d_model] 或 [N, d_model]

        输出：
        - 与 x 同形状的归一化结果
        """
        # x.mean(dim=-1, keepdim=True):
        # 在最后一维求均值，keepdim=True 使结果可与 x 广播。
        mean = x.mean(dim=-1, keepdim=True)

        # x.var(dim=-1, unbiased=False, keepdim=True):
        # 在最后一维求方差。unbiased=False 表示按总体方差公式（与很多深度学习实现一致）。
        var = x.var(dim=-1, unbiased=False, keepdim=True)

        # 标准化公式：(x - mean) / sqrt(var + eps)
        # 注意这里必须用 torch.sqrt（逐元素对张量开方），
        # 不能用 math.sqrt（math.sqrt 只适用于 Python 标量）。
        out = (x - mean) / torch.sqrt(var + self.eps)

        # 仿射变换：y = gamma * out + beta
        # gamma/beta 的形状是 [d_model]，会自动广播到前面维度。
        out = self.gamma * out + self.beta
        return out


In [3]:
# ========================
# 单元测试（LayerNorm 逻辑正确性验证）
# ========================

print("\n[测试开始] layernorm.ipynb - 自定义 LayerNorm 标准单元测试")
print("说明：本测试会验证初始化、输出形状、数值正确性、与 PyTorch 官方实现对齐、梯度可回传等关键点。")

test_results = []


def run_test(test_name, fn):
    print(f"\n[测试项] {test_name}")
    try:
        fn()
        print(f"[结果] {test_name}: 通过")
        test_results.append((test_name, True, ""))
    except Exception as e:
        print(f"[结果] {test_name}: 失败")
        print(f"[错误信息] {type(e).__name__}: {e}")
        test_results.append((test_name, False, f"{type(e).__name__}: {e}"))


def test_init_params_shape_and_value():
    print("  - 步骤1：实例化 LayerNorm(d_model=8)")
    ln = LayerNorm(d_model=8)

    print("  - 步骤2：检查 gamma/beta 的形状")
    assert tuple(ln.gamma.shape) == (8,), f"gamma 形状错误: {tuple(ln.gamma.shape)}"
    assert tuple(ln.beta.shape) == (8,), f"beta 形状错误: {tuple(ln.beta.shape)}"

    print("  - 步骤3：检查 gamma 初值全1、beta 初值全0")
    assert torch.allclose(ln.gamma.detach(), torch.ones(8)), "gamma 初值不是全1"
    assert torch.allclose(ln.beta.detach(), torch.zeros(8)), "beta 初值不是全0"


def test_output_shape_consistency():
    print("  - 步骤1：构造输入 x，形状 [4, 6, 8]")
    x = torch.randn(4, 6, 8)
    ln = LayerNorm(d_model=8)

    print("  - 步骤2：前向计算")
    y = ln(x)

    print("  - 步骤3：检查输出形状与输入一致")
    assert tuple(y.shape) == tuple(x.shape), f"输出形状 {tuple(y.shape)} 与输入 {tuple(x.shape)} 不一致"


def test_zero_mean_unit_var_when_default_affine():
    print("  - 步骤1：构造输入并前向计算（gamma=1, beta=0）")
    x = torch.randn(3, 5, 8)
    ln = LayerNorm(d_model=8, eps=1e-10)
    y = ln(x)

    print("  - 步骤2：沿最后一维计算输出均值和方差")
    y_mean = y.mean(dim=-1)
    y_var = y.var(dim=-1, unbiased=False)

    print("  - 步骤3：检查均值接近0、方差接近1")
    assert torch.allclose(y_mean, torch.zeros_like(y_mean), atol=1e-5), "归一化后均值未接近0"
    assert torch.allclose(y_var, torch.ones_like(y_var), atol=1e-4), "归一化后方差未接近1"


def test_match_torch_layernorm_numerically():
    print("  - 步骤1：实例化自定义 LayerNorm 与官方 nn.LayerNorm")
    d_model = 8
    x = torch.randn(2, 4, d_model)

    my_ln = LayerNorm(d_model=d_model, eps=1e-5)
    ref_ln = nn.LayerNorm(normalized_shape=d_model, eps=1e-5, elementwise_affine=True)

    print("  - 步骤2：将参数对齐（my.gamma/beta -> ref.weight/bias）")
    with torch.no_grad():
        ref_ln.weight.copy_(my_ln.gamma)
        ref_ln.bias.copy_(my_ln.beta)

    print("  - 步骤3：比较两者输出数值")
    y_my = my_ln(x)
    y_ref = ref_ln(x)
    assert torch.allclose(y_my, y_ref, atol=1e-6), "自定义 LayerNorm 与官方实现输出不一致"


def test_backward_gradients_exist():
    print("  - 步骤1：构造 requires_grad=True 的输入")
    x = torch.randn(2, 3, 8, requires_grad=True)
    ln = LayerNorm(d_model=8)

    print("  - 步骤2：前向 + 构造标量损失 + 反向传播")
    y = ln(x)
    loss = y.sum()
    loss.backward()

    print("  - 步骤3：检查输入和参数梯度存在且形状正确")
    assert x.grad is not None, "x.grad 不应为 None"
    assert ln.gamma.grad is not None, "gamma.grad 不应为 None"
    assert ln.beta.grad is not None, "beta.grad 不应为 None"
    assert tuple(ln.gamma.grad.shape) == (8,), f"gamma.grad 形状错误: {tuple(ln.gamma.grad.shape)}"
    assert tuple(ln.beta.grad.shape) == (8,), f"beta.grad 形状错误: {tuple(ln.beta.grad.shape)}"


run_test("初始化参数形状与初值测试", test_init_params_shape_and_value)
run_test("前向输出形状一致性测试", test_output_shape_consistency)
run_test("默认仿射参数下均值方差测试", test_zero_mean_unit_var_when_default_affine)
run_test("与官方 nn.LayerNorm 数值一致性测试", test_match_torch_layernorm_numerically)
run_test("反向传播梯度可用性测试", test_backward_gradients_exist)

print("\n[测试汇总]")
passed = sum(1 for _, ok, _ in test_results if ok)
failed = len(test_results) - passed
for name, ok, err in test_results:
    status = "通过" if ok else "失败"
    print(f"- {name}: {status}")
    if err:
        print(f"  错误详情: {err}")

print(f"总计: {len(test_results)} 项, 通过: {passed}, 失败: {failed}")
if failed == 0:
    print("[最终结果] 所有 LayerNorm 测试通过：实现逻辑正确。")
else:
    raise AssertionError("存在测试失败，请根据上方错误详情逐项排查。")



[测试开始] layernorm.ipynb - 自定义 LayerNorm 标准单元测试
说明：本测试会验证初始化、输出形状、数值正确性、与 PyTorch 官方实现对齐、梯度可回传等关键点。

[测试项] 初始化参数形状与初值测试
  - 步骤1：实例化 LayerNorm(d_model=8)
  - 步骤2：检查 gamma/beta 的形状
  - 步骤3：检查 gamma 初值全1、beta 初值全0
[结果] 初始化参数形状与初值测试: 通过

[测试项] 前向输出形状一致性测试
  - 步骤1：构造输入 x，形状 [4, 6, 8]
  - 步骤2：前向计算
  - 步骤3：检查输出形状与输入一致
[结果] 前向输出形状一致性测试: 通过

[测试项] 默认仿射参数下均值方差测试
  - 步骤1：构造输入并前向计算（gamma=1, beta=0）
  - 步骤2：沿最后一维计算输出均值和方差
  - 步骤3：检查均值接近0、方差接近1
[结果] 默认仿射参数下均值方差测试: 通过

[测试项] 与官方 nn.LayerNorm 数值一致性测试
  - 步骤1：实例化自定义 LayerNorm 与官方 nn.LayerNorm
  - 步骤2：将参数对齐（my.gamma/beta -> ref.weight/bias）
  - 步骤3：比较两者输出数值
[结果] 与官方 nn.LayerNorm 数值一致性测试: 通过

[测试项] 反向传播梯度可用性测试
  - 步骤1：构造 requires_grad=True 的输入
  - 步骤2：前向 + 构造标量损失 + 反向传播
  - 步骤3：检查输入和参数梯度存在且形状正确
[结果] 反向传播梯度可用性测试: 通过

[测试汇总]
- 初始化参数形状与初值测试: 通过
- 前向输出形状一致性测试: 通过
- 默认仿射参数下均值方差测试: 通过
- 与官方 nn.LayerNorm 数值一致性测试: 通过
- 反向传播梯度可用性测试: 通过
总计: 5 项, 通过: 5, 失败: 0
[最终结果] 所有 LayerNorm 测试通过：实现逻辑正确。
